In [ ]:
import random
import numpy as np
from datasets import load_dataset, get_dataset_config_names
from PIL import Image
import io
import pandas as pd

# ============================================================================
# 📄 VDR_Military 데이터셋 활용 초급 AI 실습 스크립트
# 🤖 데이터셋 명: racineai/VDR_Military (군사/문서 검색 증거)
# 💡 의미 및 설명: 이 데이터셋은 PDF 문서를 여러 페이지로 쪼개고, 해당 내용에 대한
#   '질문(query)'과 '답변이 될 수 있는 문서 조각(extracted_page_text, image)'을
#   짝지은 것입니다. 군사, 법률, 기술 매뉴얼 등 전문 문서에서 정보를 검색하는
#   '문서 검색 증거 (Document Search Evidence, DSE)' 분야에 사용됩니다.
# 🚀 목표: 주어진 질문(query)을 바탕으로, 가장 관련성이 높은 문맥(page_text)이나
#    이미지(image) 조각을 찾아내는 '검색(Retrieval)' AI를 이해하는 실습입니다.
# ============================================================================

# --- 설정 값 ---
DATASET_NAME = "racineai/VDR_Military"
SAMPLE_COUNT = 50 # 처음 50개의 샘플만 사용하여 실습의 부담을 줄입니다.
# -----------------

# 1. 데이터셋 로딩 준비 및 Config 확인
try:
    # 데이터셋의 사용 가능한 Config 목록을 먼저 확인합니다.
    configs = get_dataset_config_names(DATASET_NAME)
    print(f"✅ 사용 가능한 Config 목록: {configs}")
    
    # 가장 첫 번째 Config를 기본으로 사용합니다.
    selected_config = configs[0]
    print(f"🔗 기본 사용 Config: {selected_config}")

except Exception as e:
    print(f"ℹ️ Config 목록 확인 중 오류가 발생했습니다: {e}")
    selected_config = None

# 2. 스트리밍 로딩을 시도하고, 실패 시 예외 처리를 합니다.
dataset = None
try:
    # 💡 시도 1: 스트리밍 모드 (메모리 효율적)
    print("\n--- 🚀 스트리밍 모드로 데이터셋 로드를 시도합니다 (전체 데이터셋 접근) ---")
    # split='train'을 명시하고 streaming=True로 로드합니다.
    dataset = load_dataset(DATASET_NAME, name=selected_config, split='train', streaming=True)
    print("✅ 스트리밍 모드 로드 성공! 매우 큰 데이터셋에 대한 접근이 가능합니다.")

except Exception as e:
    # 💡 실패 시: 예외 처리하여 소량의 일반 데이터셋을 다운로드합니다.
    print(f"\n⚠️ 스트리밍 모드 로드 실패 또는 환경 제약으로 인해 예외 발생: {e}")
    print("✨ 소량의 일반 데이터셋을 다운로드하여 실습을 진행합니다.")
    try:
        dataset = load_dataset(DATASET_NAME, name=selected_config, split='test')
    except Exception as e_small:
        print(f"❌ 소량 다운로드에도 실패했습니다. 설정을 확인해주세요. ({e_small})")


# 3. 샘플 데이터셋 준비 (핵심 로직: 스트리밍 체크)
sampled_dataset = None
if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋 (IterableDataset)
    print(f"\n--- 🔬 총 {SAMPLE_COUNT}개의 샘플을 스트리밍 방식으로 가져옵니다. ---")
    sample_iterator = dataset.take(SAMPLE_COUNT)
    # 샘플 데이터를 list 형태로 변환하여 전체 실습에 사용합니다.
    sampled_dataset = list(sample_iterator)
    print("✅ 샘플 데이터셋 준비 완료.")
else:
    # 일반 데이터셋 (Dataset)
    # 전체 데이터를 list로 변환하여 사용합니다.
    sampled_dataset = dataset.select(range(min(SAMPLE_COUNT, len(dataset))))
    print("✅ 샘플 데이터셋 준비 완료 (일반 Dataset 모드).")


# 4. 데이터셋 탐색 및 분석 (Quantitative Analysis)
def analyze_data(data_list):
    """로드된 샘플 데이터 리스트를 기반으로 간단한 통계 분석을 수행합니다."""
    print("\n" + "="*70)
    print("🔎 [1단계] 데이터 구조 탐색 및 정량적 분석")
    print("="*70)
    
    # 4-1. 전반적인 개요
    print(f"💡 분석 샘플 개수: {len(data_list)} 건")
    print("🏷️ 주요 필드: 'query'(질문), 'language'(언어), 'extracted_page_text'(문맥/추출 텍스트), 'image'(이미지)")
    
    # 4-2. 언어 분포 분석 (가장 쉬운 통계)
    language_counts = {}
    for sample in data_list:
        lang = sample.get("language", "N/A")
        language_counts[lang] = language_counts.get(lang, 0) + 1
    
    print("\n🌎 언어별 빈도수 (상위 5개):")
    sorted_languages = sorted(language_counts.items(), key=lambda item: item[1], reverse=True)
    for lang, count in sorted_languages[:5]:
        print(f"   - {lang}: {count} 회")

    # 4-3. 결측치 확인 (Query 존재 여부)
    query_present = sum(1 for sample in data_list if 'query' in sample and sample['query'] is not None)
    print(f"\n❓ 'query' 필드가 존재하는 샘플 비율: {query_present / len(data_list) * 100:.2f}%")


def simulate_retrieval(data_list):
    """
    실습 2: 검색 시뮬레이션 (Retrieval Simulation)
    주어진 질문과 가장 유사한 문맥을 찾는 과정을 모방합니다.
    """
    print("\n" + "="*70)
    print("🔍 [2단계] 문서 검색 시뮬레이션 (Retrieval Simulation)")
    print("="*70)
    
    # 테스트할 임의의 '질문'을 첫 샘플에서 가져옵니다.
    if not data_list:
        print("❌ 데이터가 없어 시뮬레이션을 수행할 수 없습니다.")
        return
    
    # 첫 번째 샘플의 query를 사용해 시뮬레이션 질문을 정의합니다.
    target_query = data_list[0].get("query")
    if not target_query:
        print("❌ 첫 번째 샘플에서 유효한 질문을 가져올 수 없습니다.")
        return
        
    print(f"💡 [테스트 질문]: '{target_query[:50]}...'")
    print("✅ 시스템이 이 질문과 가장 관련 깊은 '문맥'을 검색합니다.")
    
    # 가장 관련성이 높은 상위 3개의 문맥을 찾습니다. (간단한 길이 기반 유사도 계산)
    # 실제 AI는 임베딩(Embedding)을 사용하지만, 초보자 실습을 위해 길이와 키워드를 결합하여 점수를 매깁니다.
    
    def calculate_score(query, text):
        """질문과 텍스트의 간접적인 연관성 점수를 계산합니다."""
        if not text: return 0
        score = 0
        # 1. 길이 적합성 (너무 짧거나 길지 않은 것이 좋음)
        length_match_score = 1.0 / (1 + abs(len(text) - 100)) # 예시 길이 100 기준
        score += length_match_score * 0.3
        
        # 2. 키워드 일치 (간단하게 쿼리 키워드를 포함하는지 체크)
        query_keywords = set(query.lower().split()[:3]) # 처음 3개 단어만 키워드로 사용
        for keyword in query_keywords:
            if keyword in text.lower():
                score += 0.2 # 키워드가 발견될 때마다 점수 증가
        return score

    # 모든 샘플에 대해 점수를 계산하고 정렬합니다.
    scored_samples = []
    for i, sample in enumerate(data_list):
        text = sample.get("extracted_page_text", "")
        score = calculate_score(target_query, text)
        scored_samples.append((score, sample))
    
    # 점수가 높은 순서대로 정렬
    scored_samples.sort(key=lambda x: x[0], reverse=True)
    
    print("\n🥇 검색 결과 (상위 3개):")
    for rank, (score, sample) in enumerate(scored_samples[:3]):
        # 문맥이 너무 길면 자르고 보기 좋게 만듭니다.
        snippet = sample.get("extracted_page_text", "No text available.")[:100] + "..."
        print(f"  [{rank + 1}] (Score: {score:.2f})")
        print(f"      📜 문맥 내용: {snippet}")
        print(f"      💬 원본 쿼리: {sample.get('query', 'N/A')}")
        display(sample.get('image'))


# 5. 메인 실행 흐름
if __name__ == "__main__":
    
    if sampled_dataset:
        # 1단계: 데이터 구조 및 통계 분석
        analyze_data(sampled_dataset)
        
        # 2단계: 검색 시뮬레이션
        simulate_retrieval(sampled_dataset)
        
        print("\n" + "="*70)
        print("✨ 실습 완료!")
        print("이 스크립트는 DSE(문서 검색 증거)의 기본 원리인 '질문-문맥 매칭' 과정을 이해하기 위한 시뮬레이션입니다.")
        print("실제 AI에서는 '임베딩'을 사용해 의미적 유사도를 계산하지만, 여기서는 간단한 키워드 매칭으로 이를 모방했습니다.")
        print("="*70)
    else:
        print("\n[오류] 실습을 진행할 데이터셋을 로드하지 못했습니다.")